In [1]:
import dill
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [3]:
df = pd.read_excel(r"..\Data\Original_Data\Motot_Type_YWNC-203.xlsx")

In [4]:
with open("full_pipeline_dill.pkl", "rb") as f:
    full_pipeline = dill.load(f)

In [7]:
final_df = full_pipeline.named_steps["preprocess"].fit_transform(df)

C:\Users\devan\AppData\Local\Temp\ipykernel_13208\3369979671.py:7: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
C:\Users\devan\AppData\Local\Temp\ipykernel_13208\3369979671.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.




In [11]:
full_pipeline.named_steps["rf"].fit(final_df.drop(columns=["HOURLY_KWH_capped", "HOURLY_KWH"], errors="ignore"), final_df['HOURLY_KWH_capped'])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsam

In [12]:
# -----------------------------
# 1. Identify last timestamp
# -----------------------------
last_time = final_df.index.max()
print("Last timestamp:", last_time)

# -----------------------------
# 2. Create next 7 days hourly timestamps
# -----------------------------
HORIZON = 24 * 7  # 168 hours

next_times = pd.date_range(
    start=last_time + pd.Timedelta(hours=1),
    periods=HORIZON,
    freq="h"
)

next_X = pd.DataFrame(index=next_times)
next_X["hour"] = next_X.index.hour
next_X["weekday"] = next_X.index.weekday

hour = next_X["hour"]
weekday = next_X["weekday"]

# -----------------------------
# 3. Prepare historical stats (PAST DATA ONLY)
# -----------------------------
hist_df = final_df.loc[final_df.index < next_times[0]].copy()
hist_df["hour"] = hist_df.index.hour
hist_df["weekday"] = hist_df.index.weekday

weekday_hour_means = (
    hist_df
    .groupby(["weekday", "hour"])[["AVG_CURRENT", "AVG_V_LN"]]
    .mean()
)

hour_means = (
    hist_df
    .groupby("hour")[["AVG_CURRENT", "AVG_V_LN"]]
    .mean()
)

# -----------------------------
# 4. Populate AVG_CURRENT & AVG_V_LN
# -----------------------------
next_X = next_X.merge(
    weekday_hour_means,
    left_on=["weekday", "hour"],
    right_index=True,
    how="left"
)

for col in ["AVG_CURRENT", "AVG_V_LN"]:
    next_X[col] = next_X[col].fillna(
        next_X["hour"].map(hour_means[col])
    )

next_X[["AVG_CURRENT", "AVG_V_LN"]] = (
    next_X[["AVG_CURRENT", "AVG_V_LN"]]
    .fillna(hist_df[["AVG_CURRENT", "AVG_V_LN"]].mean())
)
# -----------------------------
# 5. Long Gap Flag
# -----------------------------


# -----------------------------
# 5. Power proxy
# -----------------------------
next_X["power_proxy"] = (
    next_X["AVG_CURRENT"] * next_X["AVG_V_LN"]
)

# -----------------------------
# 6. Cyclical time encoding
# -----------------------------
next_X["hour_sin"] = np.sin(2 * np.pi * hour / 24)
next_X["hour_cos"] = np.cos(2 * np.pi * hour / 24)
next_X["weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
next_X["weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

# -----------------------------
# 7. Lag features (STATIC – LAST KNOWN)
# -----------------------------
hist_df["kwh_lag_1"]   = final_df['HOURLY_KWH'].shift(1)
hist_df["kwh_lag_8"]   = final_df['HOURLY_KWH'].shift(8)
hist_df["kwh_lag_24"]  = final_df['HOURLY_KWH'].shift(24)
hist_df["kwh_lag_168"] = final_df['HOURLY_KWH'].shift(168)

lag_cols  = ["kwh_lag_24", "kwh_lag_168"]
weekday_hour_lag_means = (
    hist_df
    .groupby(["weekday", "hour"])[lag_cols]
    .mean()
)
hour_lag_means = (
    hist_df
    .groupby("hour")[lag_cols]
    .mean()
)

# Lag features
next_X = next_X.merge(
    weekday_hour_lag_means,
    left_on=["weekday", "hour"],
    right_index=True,
    how="left"
)

# Lag fallbacks
for col in lag_cols:
    next_X[col] = next_X[col].fillna(
        next_X["hour"].map(hour_lag_means[col])
    )

next_X[lag_cols] = next_X[lag_cols].fillna(
    hist_df[lag_cols].mean()
)

# -----------------------------
# 8. Rolling statistics
# -----------------------------
hist_df["kwh_roll_2"]  = final_df['HOURLY_KWH'].rolling(2).mean()
hist_df["kwh_roll_24"] = final_df['HOURLY_KWH'].rolling(24).mean()

roll_cols = ["kwh_roll_2", "kwh_roll_24"]
weekday_hour_roll_means = (
    hist_df
    .groupby(["weekday", "hour"])[roll_cols]
    .mean()
)
hour_roll_means = (
    hist_df
    .groupby("hour")[roll_cols]
    .mean()
)

# Rolling features
next_X = next_X.merge(
    weekday_hour_roll_means,
    left_on=["weekday", "hour"],
    right_index=True,
    how="left"
)

# Rolling fallbacks
for col in roll_cols:
    next_X[col] = next_X[col].fillna(
        next_X["hour"].map(hour_roll_means[col])
    )

next_X[roll_cols] = next_X[roll_cols].fillna(
    hist_df[roll_cols].mean()
)

# -----------------------------
# 10. Gap & activity flags
# -----------------------------
# next_X["time_since_gap"] = final_df["time_since_gap"].iloc[-1] + 1
# next_X["time_since_gap_log"] = final_df["time_since_gap_log"].iloc[-1] + 1
next_X["low_activity_detected"] = final_df["low_activity_detected"].iloc[-1]
next_X["spike_detected"] = final_df["spike_detected"].iloc[-1]

# -----------------------------
# 11. Align feature order
# -----------------------------
feature_cols = (
    final_df
    .drop(columns=["HOURLY_KWH_capped", "HOURLY_KWH"], errors="ignore")
    .columns
    .tolist()
)
next_X = next_X[feature_cols]

# -----------------------------
# 12. Predict (Random Forest)
# -----------------------------
next_pred = full_pipeline.named_steps["rf"].predict(next_X)

# -----------------------------
# 13. Forecast DataFrame
# -----------------------------
forecast_df = pd.DataFrame(
    {"Predicted": next_pred},
    index=next_X.index
)


Last timestamp: 2025-12-29 00:00:00


In [13]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=forecast_df.index,
    y=forecast_df["Predicted"],
    mode="lines",
    name="Next 7 Days Forecast",
    line=dict(width=2)
))

fig.update_layout(
    title="Next 7 Days Hourly Energy Forecast (Random Forest)",
    xaxis_title="Time",
    yaxis_title="Energy / Load",
    template="plotly_white"
)

fig.show()

In [14]:
daily_forecast = (
    forecast_df
    .resample("D")
    .sum()
)

fig.add_trace(go.Bar(
    x=daily_forecast.index,
    y=daily_forecast["Predicted"],
    name="Daily Energy (kWh)",
    opacity=0.6
))

In [ ]:
final_df.tail()

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,power_proxy,hour,weekday,hour_sin,hour_cos,weekday_sin,weekday_cos,kwh_lag_24,kwh_lag_168,kwh_roll_2,kwh_roll_24,low_activity_detected,spike_detected,HOURLY_KWH_capped
2025-12-29 04:00:00,0.154,0.555087,239.906783,133.169126,4,0,0.866025,5.000000e-01,0.0,1.0,1.416,0.150,0.1525,0.597917,0,0,0.155
2025-12-29 05:00:00,0.156,0.554348,239.902261,132.989297,5,0,0.965926,2.588190e-01,0.0,1.0,1.412,0.155,0.1550,0.545583,0,0,0.156
2025-12-29 06:00:00,0.148,0.560955,238.226591,133.634289,6,0,1.000000,6.123234e-17,0.0,1.0,1.234,0.149,0.1520,0.500333,0,0,0.155
2025-12-29 07:00:00,1.294,4.616652,232.996217,1075.662494,7,0,0.965926,-2.588190e-01,0.0,1.0,1.246,0.943,0.7210,0.502333,0,0,1.294
2025-12-29 08:00:00,0.640,4.465818,231.181091,1032.412719,8,0,0.866025,-5.000000e-01,0.0,1.0,1.392,1.473,0.9670,0.471000,0,0,0.640
